# [센서 실습 6] 초음파 거리 센서 – HC-SR04

초음파 센서 **HC-SR04**의 에코 펄스 시간을 재서 거리를 계산하고, 후방 주차 보조 경고 시스템을 만들어 봅니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
HC-SR04는 박쥐처럼 **초음파(40kHz)** 를 쏘고 물체에 맞고 돌아오는 시간을 잽니다.
1. TRIG 핀에 **10µs** 길이의 HIGH 펄스를 줌 → 초음파 8개 발사
2. 발사와 동시에 ECHO 핀이 HIGH가 되고, 반사파가 돌아오면 LOW가 됨
3. **ECHO가 HIGH였던 시간 = 왕복 시간**

$$\text{거리} = \frac{\text{왕복 시간} \times \text{음속}}{2}, \qquad \text{음속} \approx 331.3 + 0.606 \times \text{기온}(°C)\ \text{m/s}$$

20°C에서 음속은 약 343m/s → 1cm를 왕복하는 데 약 58µs가 걸립니다.

### 부품 사양
| 항목 | 내용 |
|---|---|
| 측정 거리 | 2 ~ 400 cm |
| 정확도 | 약 ±3 mm |
| 측정 각도 | 약 15° |
| 전원 | 5V |
| 출력 | ECHO 펄스 폭 (5V 레벨) |

## 2. 라즈베리파이 연결 방법
**준비물**: HC-SR04, 저항 1kΩ·2kΩ 각 1개(전압 분배), 점퍼선

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| HC-SR04 VCC | 5V (2번 핀) | 반드시 5V |
| HC-SR04 GND | GND (6번 핀) |  |
| HC-SR04 TRIG | GPIO23 (16번 핀) | 3.3V 신호로도 동작 |
| HC-SR04 ECHO → 1kΩ → GPIO24 | GPIO24 (18번 핀) | GPIO24 ↔ GND 사이에 2kΩ |

> ⚠️ **ECHO 핀은 5V를 출력**합니다. 라즈베리파이 GPIO는 3.3V까지만 견디므로 반드시 **1kΩ/2kΩ 전압 분배**로 낮춰야 합니다. (5V × 2/3 ≈ 3.3V)

**인터페이스 활성화**
- **GPIO**: 별도 설정 없이 바로 사용 가능

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install gpiozero
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
에코 펄스 시간(초)을 거리(cm)로 바꾸는 식입니다. 기온을 넣으면 음속을 보정합니다.

In [ ]:
def echo_to_distance_cm(pulse_s, temp_c=20.0):
    """에코 펄스 시간(초) → 거리(cm)"""
    if pulse_s is None:
        return None
    sound_speed = 331.3 + 0.606 * temp_c            # m/s
    return pulse_s * sound_speed * 100 / 2

print(f"에코 5.83ms → {echo_to_distance_cm(0.00583):.1f} cm")

### 4-2. 실제 센서 클래스
TRIG에 10µs 펄스를 주고, ECHO가 HIGH인 시간을 직접 잽니다. 30ms 안에 응답이 없으면 측정 범위 밖으로 처리합니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealHCSR04:
    def __init__(self, trig=23, echo=24):
        from gpiozero import DigitalOutputDevice, DigitalInputDevice
        self.trig = DigitalOutputDevice(trig)
        self.echo = DigitalInputDevice(echo)

    def read(self, timeout=0.03):
        self.trig.on(); time.sleep(0.00001); self.trig.off()       # 10µs 트리거
        t0 = time.perf_counter()
        while not self.echo.value:                                 # 에코 시작 대기
            if time.perf_counter() - t0 > timeout:
                return {"pulse_s": None, "distance_cm": None}
        start = time.perf_counter()
        while self.echo.value:                                     # 에코 끝 대기
            if time.perf_counter() - start > timeout:
                return {"pulse_s": None, "distance_cm": None}
        pulse = time.perf_counter() - start
        return {"pulse_s": pulse, "distance_cm": echo_to_distance_cm(pulse)}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 항목 | 가정한 값 | 근거 |
|---|---|---|
| 상황 | 자동차 뒤범퍼의 센서, 후진 주차 | 주차 보조 장치 |
| 거리 변화 | 250cm에서 출발 → 15cm에서 정지 (감속하며 접근) | 실제 후진 주차 속도 |
| 펄스 지터 | ±20µs (약 ±3mm) | 데이터시트 정확도 |
| 이상값 | 5% 확률로 엉뚱한 반사(엉뚱한 거리) | 바닥·옆 물체에서 반사되는 현상 |
| 범위 초과 | 400cm 이상이면 응답 없음(`None`) | 최대 측정 거리 |

가상 센서는 실제 거리로부터 **에코 펄스 시간**을 만들어 변환식에 넣습니다.

In [ ]:
class VirtualHCSR04:
    def __init__(self, dt=0.2, seed=6):
        self.dt, self.t = dt, 0.0
        self.rng = np.random.default_rng(seed)

    def _true_distance(self):
        # 20초 동안 250cm → 15cm로 감속하며 접근 후 정지
        return 15 + 235 * max(0.0, 1 - self.t / 20) ** 1.5

    def read(self):
        d = self._true_distance()
        if self.rng.random() < 0.05:
            d = self.rng.uniform(5, 300)                             # 엉뚱한 반사
        self.t += self.dt
        if d > 400:
            return {"pulse_s": None, "distance_cm": None}
        pulse = 2 * (d / 100) / 343.4 + self.rng.normal(0, 20e-6)   # 왕복 시간 + 지터
        return {"pulse_s": pulse, "distance_cm": echo_to_distance_cm(pulse)}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualHCSR04(dt=0.2)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
0.2초 간격으로 25초 동안 측정합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 125               # 측정 횟수
INTERVAL = 0.2               # 측정 간격(초)

sensor = RealHCSR04() if USE_REAL_SENSOR else VirtualHCSR04(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
1. **중앙값 필터**(최근 5개 중 가운데 값)로 엉뚱한 반사값을 제거합니다. 평균과 달리 튀는 값에 흔들리지 않습니다.
2. 거리에 따라 경고 단계를 정합니다.

| 거리 | 단계 | 경고음 |
|---|---|---|
| 100cm 이상 | 안전 | 없음 |
| 50 ~ 100cm | 주의 | 삐- 삐- |
| 30 ~ 50cm | 경고 | 삐삐삐 |
| 30cm 미만 | 위험 | 삐------ |

In [ ]:
df["filtered"] = df["distance_cm"].rolling(5, center=True, min_periods=1).median()

def level(d):
    return "안전" if d >= 100 else "주의" if d >= 50 else "경고" if d >= 30 else "위험"
df["level"] = df["filtered"].apply(level)

prev = None
for _, r in df.iterrows():
    if r["level"] != prev:
        print(f"{r['time_s']:5.1f}초  {r['filtered']:6.1f} cm  → {r['level']}")
        prev = r["level"]
print(f"\n튀는 값 제거 전 최댓값 {df['distance_cm'].max():.0f}cm / 제거 후 {df['filtered'].max():.0f}cm")

## 8. 시각화

In [ ]:
plt.figure(figsize=(10, 4.5))
plt.plot(df["time_s"], df["distance_cm"], ".", color="gray", ms=5, label="원시 측정값")
plt.plot(df["time_s"], df["filtered"], "k-", lw=2, label="중앙값 필터")
for y, name in [(100, "주의"), (50, "경고"), (30, "위험")]:
    plt.axhline(y, color="gray", ls="--", lw=0.8); plt.text(25.3, y, name, va="center")
plt.title("HC-SR04 후방 주차 보조"); plt.xlabel("시간 (초)"); plt.ylabel("거리 (cm)")
plt.grid(alpha=0.3); plt.legend(); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "초음파거리센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "HC-SR04",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/distance", payload)
# client.disconnect()

## 직접 해 보기
- 기온을 0°C와 35°C로 바꿔 넣으면 같은 펄스로 계산한 거리가 얼마나 달라지나요?
- 중앙값 필터 대신 이동 평균(`rolling(5).mean()`)을 쓰면 어떻게 되는지 비교해 보세요.
- 위험 단계가 2초 이상 이어지면 '자동 브레이크' 메시지를 출력해 보세요.